# Switch to a local model (Ollama)

Ollama runs open-weight models in the codespace behind an OpenAI-compatible API: we change only `base_url` and `model`, then evaluate again.

In this notebook, we will:
1. Check that Ollama runs.
2. Point the OpenAI client to `localhost:11434/v1`.
3. Rerun the review classification on 20 reviews.
4. Compare accuracy, speed and cost with `gpt-6-luna`.

**Time:** about 30 minutes, self-study. Tip: a 4-core codespace is faster.

## Libraries and settings

Both models are called directly with the OpenAI library, without cache, to measure the real speed.

In [ ]:
# Libraries
import os
import re
import time
import platform
import warnings
from datetime import datetime
from platform import python_version

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import accuracy_score

# Ignore warnings
warnings.filterwarnings('ignore')

# Settings
MODEL = 'gpt-6-luna'
PRICE_INPUT_PER_M = 0.10        # USD per 1M input tokens
LOCAL_MODEL = 'qwen3:1.7b'

# Key from a local .env file (only needed outside Codespaces)
load_dotenv('../.env')
print('API key found:', 'OPENAI_API_KEY' in os.environ)

# Show current working directory
print(os.getcwd())

## Step 1: Check that Ollama runs

Installs Ollama if missing (about 2 minutes), starts the server and downloads the model (about 1.4 GB, once).

In [ ]:
!which ollama || curl -fsSL https://ollama.com/install.sh | sh
!pgrep -x ollama > /dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)
!sleep 3 && ollama pull {LOCAL_MODEL} && ollama list

## Step 2: Point the OpenAI client to localhost:11434/v1

Ollama ignores the key, but the client needs one.

In [ ]:
local = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
api = OpenAI(max_retries=8)

r = local.chat.completions.create(
    model=LOCAL_MODEL,
    messages=[{'role': 'user', 'content': 'Name one typical dish in the IKEA restaurant. One sentence.'}],
)
print(r.choices[0].message.content)

## Step 3: Rerun the review classification on 20 reviews

20 reviews with silver labels from `03_Classify_Reviews`, the same one-word prompt for both models.

In [ ]:
# Reviews (data from Moodle, see README: Data) and silver labels from 03_Classify_Reviews
df = pd.read_csv('../data/ikea_reviews.csv', sep=',', encoding='utf-8')
silver = pd.read_csv('../data/processed/silver_labels.csv')
test = silver.merge(df, on='review_id').sample(20, random_state=42)

PROMPT = 'Positive, negative or mixed? Answer with one word.\n'

def first_label(text):
    """First label word in the answer, without the <think> part of local models."""
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).lower()
    found = re.findall(r'positive|negative|mixed', text)
    return found[0] if found else 'unparsed'

In [ ]:
# Local model: Chat Completions API
labels, start = [], time.time()
for t in test['text']:
    r = local.chat.completions.create(model=LOCAL_MODEL, messages=[{'role': 'user', 'content': PROMPT + t}])
    labels.append(first_label(r.choices[0].message.content))
test['local'] = labels
local_seconds = (time.time() - start) / len(test)

In [ ]:
# API model: Responses API
labels, tokens, start = [], 0, time.time()
for t in test['text']:
    r = api.responses.create(model=MODEL, input=PROMPT + t)
    labels.append(first_label(r.output_text))
    tokens += r.usage.input_tokens
test['api'] = labels
api_seconds = (time.time() - start) / len(test)

## Step 4: Compare accuracy, speed and cost

In [ ]:
pd.DataFrame(
    {
        LOCAL_MODEL: [accuracy_score(test['silver'], test['local']), local_seconds, 0.0],
        MODEL: [accuracy_score(test['silver'], test['api']), api_seconds, tokens / 1e6 * PRICE_INPUT_PER_M],
    },
    index=['accuracy', 'seconds per review', 'API cost (USD)'],
).round(4)

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')